In [0]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

BASE_PATH = "/Workspace/Users/prarthanabg7@gmail.com/E-commerce/01_Data/"

print("Ready")

Ready


In [0]:
#Clean orders
orders_pd = pd.read_csv(BASE_PATH + "olist_orders_dataset.csv")

# Fix dates
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders_pd[col] = pd.to_datetime(orders_pd[col], errors='coerce')

# Feature engineering
orders_pd['delivery_delay_days'] = (
    orders_pd['order_delivered_customer_date'] -
    orders_pd['order_estimated_delivery_date']
).dt.days

orders_pd['is_delayed'] = (
    orders_pd['delivery_delay_days'] > 0).astype(int)

orders_pd['order_month'] = (
    orders_pd['order_purchase_timestamp']
    .dt.to_period('M').astype(str))

orders_pd['order_year'] = (
    orders_pd['order_purchase_timestamp'].dt.year)

orders_pd['order_quarter'] = (
    orders_pd['order_purchase_timestamp'].dt.quarter)

orders_pd['processing_time_days'] = (
    orders_pd['order_delivered_carrier_date'] -
    orders_pd['order_approved_at']
).dt.days

# Filter only delivered orders
orders_silver = orders_pd[
    orders_pd['order_status'] == 'delivered'].copy()

# Drop rows where delivery date missing
orders_silver = orders_silver.dropna(
    subset=['order_delivered_customer_date'])

# Drop duplicates
orders_silver = orders_silver \
    .drop_duplicates(subset=['order_id']) \
    .reset_index(drop=True)

print(f"✅ Orders:       {len(orders_silver):,} rows")
print(f"   Delayed:      {orders_silver['is_delayed'].sum():,}")
print(f"   Delay rate:   {round(orders_silver['is_delayed'].mean()*100,2)}%")
print(f"   Avg delay:    {round(orders_silver['delivery_delay_days'].mean(),2)} days")

✅ Orders:       96,470 rows
   Delayed:      6,534
   Delay rate:   6.77%
   Avg delay:    -11.88 days


In [0]:
#Clean payments
payments_pd = pd.read_csv(BASE_PATH + "olist_order_payments_dataset.csv")

# Remove zero or negative payments
payments_silver = payments_pd[
    payments_pd['payment_value'] > 0].copy()

# Some orders have multiple payment methods
# Keep highest value payment per order
payments_silver = payments_silver \
    .sort_values('payment_value', ascending=False) \
    .drop_duplicates(subset=['order_id'], keep='first') \
    .reset_index(drop=True)

print(f"✅ Payments:     {len(payments_silver):,} rows")
print(f"   Total revenue: R$ {round(payments_silver['payment_value'].sum(),2):,}")
print(f"   Avg order value: R$ {round(payments_silver['payment_value'].mean(),2)}")
print(f"\n   Payment types:\n{payments_silver['payment_type'].value_counts()}")

✅ Payments:     99,437 rows
   Total revenue: R$ 15,858,073.82
   Avg order value: R$ 159.48

   Payment types:
payment_type
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
Name: count, dtype: int64


In [0]:
#Clean Products + Translate categories
products_pd = pd.read_csv(BASE_PATH + "olist_products_dataset.csv")
category_pd = pd.read_csv(BASE_PATH + "product_category_name_translation.csv")

# Merge English category names
products_silver = products_pd.merge(
    category_pd,
    on='product_category_name',
    how='left'
)

# Fill missing categories
products_silver['product_category_name_english'] = (
    products_silver['product_category_name_english']
    .fillna('unknown'))

# Drop Portuguese column, rename English one
products_silver = products_silver \
    .drop(columns=['product_category_name']) \
    .rename(columns={
        'product_category_name_english': 'product_category'
    }) \
    .drop_duplicates(subset=['product_id']) \
    .reset_index(drop=True)

print(f"✅ Products:     {len(products_silver):,} rows")
print(f"   Categories:   {products_silver['product_category'].nunique()}")
print(f"\n   Top 5:\n{products_silver['product_category'].value_counts().head()}")

✅ Products:     32,951 rows
   Categories:   72

   Top 5:
product_category
bed_bath_table     3029
sports_leisure     2867
furniture_decor    2657
health_beauty      2444
housewares         2335
Name: count, dtype: int64


In [0]:
#Clean Customers
customers_pd = pd.read_csv(BASE_PATH + "olist_customers_dataset.csv")

customers_silver = customers_pd.copy()

# Standardize text
customers_silver['customer_state'] = (
    customers_silver['customer_state']
    .str.upper().str.strip())

customers_silver['customer_city'] = (
    customers_silver['customer_city']
    .str.title().str.strip())

customers_silver = customers_silver \
    .drop_duplicates(subset=['customer_id']) \
    .reset_index(drop=True)

print(f"✅ Customers:    {len(customers_silver):,} rows")
print(f"   States:       {customers_silver['customer_state'].nunique()}")
print(f"\n   Top 5 states:\n{customers_silver['customer_state'].value_counts().head()}")

✅ Customers:    99,441 rows
   States:       27

   Top 5 states:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64


In [0]:
#Clean Order items
items_pd = pd.read_csv(BASE_PATH + "olist_order_items_dataset.csv")

items_silver = items_pd.dropna(
    subset=['product_id', 'seller_id']).copy()

# Fix date
items_silver['shipping_limit_date'] = pd.to_datetime(
    items_silver['shipping_limit_date'], errors='coerce')

# Total value per item including freight
items_silver['item_total_value'] = (
    items_silver['price'] +
    items_silver['freight_value'])

items_silver = items_silver \
    .drop_duplicates() \
    .reset_index(drop=True)

print(f"✅ Items:        {len(items_silver):,} rows")
print(f"   Avg price:    R$ {round(items_silver['price'].mean(),2)}")
print(f"   Avg freight:  R$ {round(items_silver['freight_value'].mean(),2)}")
print(f"   Avg total:    R$ {round(items_silver['item_total_value'].mean(),2)}")

✅ Items:        112,650 rows
   Avg price:    R$ 120.65
   Avg freight:  R$ 19.99
   Avg total:    R$ 140.64


In [0]:
#Clean Reviews
reviews_pd = pd.read_csv(BASE_PATH + "olist_order_reviews_dataset.csv")

reviews_silver = reviews_pd.dropna(
    subset=['review_score']).copy()

# Valid scores only
reviews_silver = reviews_silver[
    reviews_silver['review_score'].between(1, 5)]

# Fix date
reviews_silver['review_creation_date'] = pd.to_datetime(
    reviews_silver['review_creation_date'], errors='coerce')

# One review per order — keep latest
reviews_silver = reviews_silver \
    .sort_values('review_creation_date', ascending=False) \
    .drop_duplicates(subset=['order_id'], keep='first') \
    .reset_index(drop=True)

# Add sentiment label
reviews_silver['sentiment'] = pd.cut(
    reviews_silver['review_score'],
    bins=[0, 2, 3, 5],
    labels=['Negative', 'Neutral', 'Positive']
)

print(f"✅ Reviews:      {len(reviews_silver):,} rows")
print(f"   Avg score:    {round(reviews_silver['review_score'].mean(),2)}")
print(f"\n   Score distribution:")
print(reviews_silver['review_score'].value_counts().sort_index())
print(f"\n   Sentiment:")
print(reviews_silver['sentiment'].value_counts())

✅ Reviews:      98,673 rows
   Avg score:    4.09

   Score distribution:
review_score
1    11364
2     3130
3     8133
4    19044
5    57002
Name: count, dtype: int64

   Sentiment:
sentiment
Positive    76046
Negative    14494
Neutral      8133
Name: count, dtype: int64


In [0]:
#Save all silver tables to Delta
silver_tables = {
    "silver_orders":    orders_silver,
    "silver_payments":  payments_silver,
    "silver_products":  products_silver,
    "silver_customers": customers_silver,
    "silver_items":     items_silver,
    "silver_reviews":   reviews_silver
}

print("=== WRITING SILVER DELTA TABLES ===\n")
for table_name, df in silver_tables.items():
    spark.createDataFrame(df).write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    print(f"✅ {table_name}: {len(df):,} rows written")

=== WRITING SILVER DELTA TABLES ===

✅ silver_orders: 96,470 rows written
✅ silver_payments: 99,437 rows written
✅ silver_products: 32,951 rows written
✅ silver_customers: 99,441 rows written
✅ silver_items: 112,650 rows written
✅ silver_reviews: 98,673 rows written


In [0]:
#Verify Silver Delta Tables
print("=== SILVER VERIFICATION ===\n")
for table_name in silver_tables.keys():
    count = spark.table(table_name).count()
    print(f"{table_name}: {count:,} rows")

=== SILVER VERIFICATION ===

silver_orders: 96,470 rows
silver_payments: 99,437 rows
silver_products: 32,951 rows
silver_customers: 99,441 rows
silver_items: 112,650 rows
silver_reviews: 98,673 rows


In [0]:
print("=== SILVER LAYER SUMMARY ===\n")
print(f"Delivered orders:     {len(orders_silver):,}")
print(f"Delayed orders:       {orders_silver['is_delayed'].sum():,}")
print(f"Delay rate:           {round(orders_silver['is_delayed'].mean()*100,2)}%")
print(f"Avg delay:            {round(orders_silver['delivery_delay_days'].mean(),2)} days")
print()
print(f"Total revenue:        R$ {round(payments_silver['payment_value'].sum(),2):,}")
print(f"Avg order value:      R$ {round(payments_silver['payment_value'].mean(),2)}")
print()
print(f"Unique products:      {len(products_silver):,}")
print(f"Product categories:   {products_silver['product_category'].nunique()}")
print()
print(f"Avg review score:     {round(reviews_silver['review_score'].mean(),2)}")
print(f"Positive reviews:     {(reviews_silver['sentiment']=='Positive').sum():,}")
print(f"Negative reviews:     {(reviews_silver['sentiment']=='Negative').sum():,}")

=== SILVER LAYER SUMMARY ===

Delivered orders:     96,470
Delayed orders:       6,534
Delay rate:           6.77%
Avg delay:            -11.88 days

Total revenue:        R$ 15,858,073.82
Avg order value:      R$ 159.48

Unique products:      32,951
Product categories:   72

Avg review score:     4.09
Positive reviews:     76,046
Negative reviews:     14,494
